# QnA - Customer Shopping Behaviour 2 ( SQL)

#### 1. Connect to the existing SQLite database

In [110]:
import sqlite3
import pandas as pd


conn = sqlite3.connect('shopping_behavior.db')

print("Successfully connected to 'shopping_behavior.db'!\n")


Successfully connected to 'shopping_behavior.db'!



# ==========================================
# Q1. Total revenue generated by male vs. female customers
# ==========================================

In [111]:

q1_result = pd.read_sql("""

SELECT 
    Gender, 
    SUM("Purchase Amount (USD)") AS Total_Revenue
FROM customer_transactions
    
GROUP BY Gender;
    
""", conn)

q1_result

,Gender,Total_Revenue
0,Female,75191
1,Male,157890


# ==========================================
# Q2. Customers who used a discount but spent more than the average purchase amount
# ==========================================

In [112]:

q2_result = pd.read_sql("""

SELECT  "Customer ID", Age, Gender, Category, "Purchase Amount (USD)", "Discount Applied"
FROM customer_transactions

WHERE "Discount Applied" = 'Yes' AND "Purchase Amount (USD)" > (SELECT AVG("Purchase Amount (USD)") 
FROM customer_transactions);

""", conn)

print(f"\nTotal customers who used a discount but spent more than the average purchase amount are: {len(q2_result)}")
df = q2_result
q2_result


Total customers who used a discount but spent more than the average purchase amount are: 839


,Customer ID,Age,Gender,Category,Purchase Amount (USD),Discount Applied
0,2,19,Male,Clothing,64,Yes
1,3,50,Male,Clothing,73,Yes
2,4,21,Male,Footwear,90,Yes
3,7,63,Male,Clothing,85,Yes
4,9,26,Male,Outerwear,97,Yes
...,...,...,...,...,...,...
834,1667,51,Male,Clothing,64,Yes
835,1671,22,Male,Clothing,73,Yes
836,1673,18,Male,Footwear,73,Yes
837,1674,21,Male,Clothing,62,Yes


In [113]:
# Save the data dataframe to a new CSV file
df.to_csv('SQL-customer_applied_dis_aboveAVG.csv', index=False)

# ==========================================
# Q3. Top 5 products with the highest average review rating
# ==========================================

In [114]:

q3_result = pd.read_sql("""

SELECT 
    "Item Purchased", 
    ROUND(AVG("Review Rating"), 2) AS Avg_Review_Rating
FROM customer_transactions

GROUP BY "Item Purchased"
ORDER BY Avg_Review_Rating DESC

LIMIT 5;

""", conn)

q3_result

,Item Purchased,Avg_Review_Rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.79


# ==========================================
# Q4. Compare average purchase amounts between Standard and Express Shipping
# ==========================================

## Average purchase amounts of - All shipping types

In [115]:
q4_result = pd.read_sql("""
SELECT 
    "Shipping Type", 
    ROUND(AVG("Purchase Amount (USD)"), 2) AS Avg_Purchase_Amount
FROM customer_transactions
GROUP BY "Shipping Type";
""", conn)

q4_result

,Shipping Type,Avg_Purchase_Amount
0,2-Day Shipping,60.73
1,Express,60.48
2,Free Shipping,60.41
3,Next Day Air,58.63
4,Standard,58.46
5,Store Pickup,59.89


## Average purchase amounts between Standard and Express Shipping

In [116]:

q4_result = pd.read_sql("""

SELECT 
    "Shipping Type", 
    ROUND(AVG("Purchase Amount (USD)"), 2) AS Avg_Purchase_Amount
FROM customer_transactions
    
WHERE "Shipping Type" IN ('Standard', 'Express')
GROUP BY "Shipping Type";

""", conn)

q4_result

,Shipping Type,Avg_Purchase_Amount
0,Express,60.48
1,Standard,58.46


# ==========================================
# Q5. Compare average spend and total revenue between subscribers and non-subscribers
# ==========================================

In [117]:

q5_result = pd.read_sql("""

SELECT 
       "Subscription Status", 
       COUNT("Customer ID") AS Total_Customers,
       ROUND(AVG("Purchase Amount (USD)"), 2) AS Avg_Spend,
       SUM("Purchase Amount (USD)") AS Total_Revenue
FROM customer_transactions

GROUP BY "Subscription Status";

""", conn)

q5_result

,Subscription Status,Total_Customers,Avg_Spend,Total_Revenue
0,No,2847,59.87,170436
1,Yes,1053,59.49,62645


# ==========================================
# Q6. Top 5 products with the highest percentage of purchases with discounts applied
# ==========================================

In [118]:

q6_result = pd.read_sql("""

SELECT 
       "Item Purchased", 
       ROUND(100.0 * SUM(CASE WHEN "Discount Applied" = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS "Highest percentage of purchases with discounts"
FROM customer_transactions

GROUP BY "Item Purchased"
ORDER BY "Highest percentage of purchases with discounts" DESC
    
    LIMIT 5;
""", conn)

q6_result

,Item Purchased,Highest percentage of purchases with discounts
0,Hat,50.00
1,Sneakers,49.66
2,Coat,49.07
3,Sweater,48.17
4,Pants,47.37


Seeing which products rely heavily on the discount.

# ==========================================
# Q7. Segment customers into New, Returning, and Loyal based on previous purchases
# ==========================================

In [119]:

q7_result = pd.read_sql("""

    SELECT 
        CASE 
            WHEN "Previous Purchases" == 1 THEN 'New'
            WHEN "Previous Purchases" BETWEEN 2 AND 10 THEN 'Returning'
            ELSE 'Loyal'
        END AS Customer_Segment,
        
        COUNT("Customer ID") AS Segment_Count
        
    FROM customer_transactions
    GROUP BY Customer_Segment;
    
""", conn)

q7_result

,Customer_Segment,Segment_Count
0,Loyal,3116
1,New,83
2,Returning,701


**There are lots of loyal customers in this data set.**

END: Marks the finish line for the CASE statement

Customer_Segment so you can easily reference it in your GROUP BY

# ==========================================
# Q8. Top 3 most purchased products within each category
# ==========================================

In [120]:
q8_result = pd.read_sql("""
    WITH RankedItems AS (
        SELECT Category, "Item Purchased", COUNT(*) AS Purchase_Count,
               ROW_NUMBER() OVER(PARTITION BY Category ORDER BY COUNT(*) DESC) as rn
        FROM customer_transactions
        GROUP BY Category, "Item Purchased"
    )
    SELECT Category, "Item Purchased", Purchase_Count
    FROM RankedItems
    WHERE rn <= 3;
""", conn)

q8_result

,Category,Item Purchased,Purchase_Count
0,Accessories,Jewelry,171
1,Accessories,Sunglasses,161
2,Accessories,Belt,161
3,Clothing,Pants,171
4,Clothing,Blouse,171
5,Clothing,Shirt,169
6,Footwear,Sandals,160
7,Footwear,Shoes,150
8,Footwear,Sneakers,145
9,Outerwear,Jacket,163


# ==========================================
# Q9. Are repeat buyers (more than 5 previous purchases) also likely to subscribe?
# ==========================================

In [121]:

q9_result = pd.read_sql("""
    SELECT 
        CASE WHEN "Previous Purchases" > 5 THEN 'Repeat Buyer (>5)' ELSE 'Low Frequency (<=5)' END AS Buyer_Type,
        "Subscription Status",
        COUNT("Customer ID") AS Customer_Count
    FROM customer_transactions
    GROUP BY Buyer_Type, "Subscription Status";
""", conn)

q9_result

,Buyer_Type,Subscription Status,Customer_Count
0,Low Frequency (<=5),No,329
1,Low Frequency (<=5),Yes,95
2,Repeat Buyer (>5),No,2518
3,Repeat Buyer (>5),Yes,958


Finding: Yes, repeat buyers show a slightly higher tendency to subscribe compared to low-frequency buyers, though overall subscription rates remain moderate across both groups.

Data Breakdown:

Repeat Buyers (>5 Previous Purchases): Out of 3,476 repeat customers, 958 are active subscribers, giving a subscription rate of ~27.56%.

Low-Frequency Buyers (≤5 Previous Purchases): Out of 424 customers, 95 are active subscribers, yielding a subscription rate of ~22.41%.

# ==========================================
# Q10. Revenue contribution of each age group
# ==========================================

In [122]:

q10_result = pd.read_sql("""

    SELECT 
           "Age Group", 
           SUM("Purchase Amount (USD)") AS Total_Revenue,
           ROUND(AVG("Purchase Amount (USD)"), 2) AS Avg_Spend
    FROM customer_transactions
    
    GROUP BY "Age Group"
    ORDER BY Total_Revenue DESC;
    
""", conn)

q10_result



,Age Group,Total_Revenue,Avg_Spend
0,Young Adult,62143,60.45
1,Middle-aged,59197,60.04
2,Adult,55978,59.42
3,Senior,55763,59.07


In [123]:
# Close the database connection when done
conn.close()
print("Database connection closed successfully.")

Database connection closed successfully.
